# Image-Based Damage Detection - CNN Training

This notebook trains a CNN to detect structural damage from images using transfer learning.

## Classes:
- No Damage
- Crack
- Spalling
- Corrosion
- Structural Deformation

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print(f"TensorFlow: {tf.__version__}")

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 30
CLASS_NAMES = ['no_damage', 'crack', 'spalling', 'corrosion', 'structural_deformation']
DATA_DIR = '../data/images'
MODEL_DIR = '../backend/saved_models'
os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
# Create sample data if not exists
def create_sample_images():
    print("Creating sample dataset...")
    for split in ['train', 'validation']:
        for class_name in CLASS_NAMES:
            class_dir = os.path.join(DATA_DIR, split, class_name)
            os.makedirs(class_dir, exist_ok=True)
            n_images = 50 if split == 'train' else 10
            colors = {'no_damage': (200, 200, 200), 'crack': (50, 50, 50),
                      'spalling': (150, 100, 50), 'corrosion': (150, 50, 50),
                      'structural_deformation': (100, 100, 150)}
            for i in range(n_images):
                color = colors[class_name]
                img = np.random.randint(0, 50, (224, 224, 3), dtype=np.uint8)
                img = np.clip(img + np.array(color, dtype=np.uint8), 0, 255).astype(np.uint8)
                Image.fromarray(img).save(os.path.join(class_dir, f'{class_name}_{i:03d}.jpg'))
    print("Done!")

if not os.path.exists(os.path.join(DATA_DIR, 'train')):
    create_sample_images()

In [ ]:
train_datagen = ImageDataGenerator(rescale=1./255, rotation_range=20, horizontal_flip=True)
val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    os.path.join(DATA_DIR, 'train'), target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', classes=CLASS_NAMES)

val_gen = val_datagen.flow_from_directory(
    os.path.join(DATA_DIR, 'validation'), target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', classes=CLASS_NAMES)

In [ ]:
# Build model
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
base_model.trainable = False

model = keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(len(CLASS_NAMES), activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# Train
history = model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS)

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['accuracy'], label='Train')
ax1.plot(history.history['val_accuracy'], label='Val')
ax1.set_title('Accuracy')
ax1.legend()
ax2.plot(history.history['loss'], label='Train')
ax2.plot(history.history['val_loss'], label='Val')
ax2.set_title('Loss')
ax2.legend()
plt.show()

In [ ]:
# Save model
model.save(os.path.join(MODEL_DIR, 'damage_detector.h5'))
print(f"Model saved to {MODEL_DIR}/damage_detector.h5")

In [ ]:
# Test prediction
test_batch = next(val_gen)
pred = model.predict(test_batch[0][:1])
print(f"Prediction: {CLASS_NAMES[np.argmax(pred)]} ({np.max(pred):.1%})")